# 🛡️ SecureHeal Arena: GRPO Training Notebook
### Meta PyTorch OpenEnv Hackathon India 2026

This notebook demonstrates how to train a `Qwen2.5-3B-Instruct` model on the **SecureHeal Arena** environment using **Group Relative Policy Optimization (GRPO)** via TRL and Unsloth.

**Project Links:**
- 🕹️ **Environment (HF Space):** [ravindraog/secureheal-trainer](https://huggingface.co/spaces/ravindraog/secureheal-trainer)
- 📊 **WandB Logs:** [secureheal-arena-grpo](https://wandb.ai/ravindraog/secureheal-arena)
- 🐙 **Source Code:** [GitHub Repo](https://github.com/ravindraogg/Geoalloc/tree/secureheal_arena)

In [ ]:
# 1. Install Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install openenv-core wandb

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from openenv import from_hub
import wandb

# 2. Configuration
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ENV_URL = "https://ravindraog-secureheal-trainer.hf.space"
WANDB_PROJECT = "secureheal-arena"

# 3. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Define Reward Functions
def reward_fn(prompts, completions, **kwargs):
    # Connect to the SecureHeal Arena Space to verify actions
    # In this notebook, we use the heuristic signals for speed,
    # but the environment-in-the-loop uses Tier 2 RLVR signals.
    rewards = []
    for completion in completions:
        score = 0.0
        # +0.4 for correct tool usage syntax
        if "<tool_call>" in completion and "</tool_call>" in completion:
            score += 0.4
        # +0.3 for security reasoning keywords
        if any(kw in completion.lower() for kw in ["sql injection", "vulnerability", "patch"]):
            score += 0.3
        # +0.3 for following the Alpha/Beta/Gamma protocol
        if "[AGENT" in completion:
            score += 0.3
        rewards.append(score)
    return rewards

In [ ]:
# 5. Training Setup
training_args = GRPOConfig(
    output_dir="secureheal-arena-grpo",
    per_device_train_batch_size=4,
    num_generations=8,
    max_prompt_length=512,
    max_completion_length=512,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=1,
    report_to="wandb",
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [reward_fn],
    args = training_args,
    train_dataset = dataset, # Loaded from secureheal_arena/vulnerabilities.py
)

trainer.train()

## Results Visualization
After training, check your WandB dashboard for the reward curves. A successful run should show the `reward/mean` rising as the agent learns to use the `<tool_call>` tags correctly and identifies security flaws more accurately.